<div align="center">
    <img src="../../../media/a365-agents.png" width="100%" alt="Microsoft Foundry workshop / lab / sample">
</div>

# Build a Foundry agent that is fully in sync with Agent 365 (WIP!!!)

In this lab you will:

1. **Register an Entra Agent ID** &mdash; a workload identity for your AI agent.
2. **Build a pro-code Foundry agent** with the `azure-ai-projects` SDK and two function tools.
3. **Bind** the Foundry agent to the Entra Agent ID so its runs execute under the agent identity.
4. **Author the Agent 365 manifest** in code (no hidden YAML).
5. **Publish to Agent 365** with the Microsoft 365 Agents Toolkit (`atk`).
6. **Verify the agent is fully in sync** across Entra, Foundry, and Agent 365.
7. **Apply and test five Agent 365 policies**: DLP, Conditional Access, tool allow-list, audit, lifecycle.
8. **Clean up** so re-running the notebook is idempotent.

## Architecture

```
        ┌──────────────────────┐        ┌────────────────────────┐
        │   Entra ID           │        │   Microsoft Foundry    │
        │  (Agent ID = appId)  │◀──bind─│  (Agent runtime)       │
        └──────────┬───────────┘        └────────────┬───────────┘
                   │                                 │
                   │ identity                        │ runtime
                   ▼                                 ▼
        ┌──────────────────────────────────────────────────────────┐
        │                Agent 365 (governance plane)              │
        │  manifest · DLP · Conditional Access · audit · lifecycle │
        └──────────────────────────────────────────────────────────┘
```

**Fully in sync** means the same logical agent is observable in all three planes
with consistent identifiers (`entraAgentId == manifest.identity.entraAgentId`,
`foundryAgentId == manifest.runtime.agentId`).

> **Run the main workshop first.** This lab assumes you completed
> [`src/workshop/README.md`](../../workshop/README.md) and have a populated
> [`src/workshop/.env`](../../workshop/.env).

## Prerequisites

Before you start this lab, make sure the following prerequisites are in place.

| # | Requirement | How to verify / get it |
| --- | --- | --- |
| 1 | Completed the main workshop and populated the `.env` file | See [`src/workshop/README.md`](../../workshop/README.md) |
| 2 | Microsoft 365 tenant with **Agent 365** enabled | Microsoft 365 admin center → **Agents** |
| 3 | Signed-in user has the **AI Administrator** or **Global Administrator** role | Entra admin center → **Roles & administrators** |
| 4 | Signed-in user can manage owned app registrations | Verify with `az ad signed-in-user show` |
| 5 | `node` version 20 or higher and `npm` installed | Run `node --version` |
| 6 | Microsoft 365 Agents Toolkit CLI (`atk`) installed | Installed in **Step 5** of this lab |
| 7 | Azure CLI login completed in this terminal | Run `az account show` |
| 8 | Microsoft Cognitive Services resource provider registered for the subscription | See command below |

Register the resource provider once per Azure subscription:

```bash
az provider register --namespace Microsoft.CognitiveServices

## Select the right Python kernel before you run anything

> [!IMPORTANT]
> Before executing any cell, make sure the notebook is using the **workshop's
> shared virtual environment** — not the system Python and not a fresh
> auto-created kernel. All other labs in this repo use the same `.venv` so
> packages installed once are reused everywhere.

1. Click the kernel picker in the top-right of this notebook (it may say
   *Select Kernel* or show a different interpreter).
2. Choose **Python Environments…** → **`.venv (Python 3.x)`** located at
   the repo root: `/workspaces/Microsoft-Foundry/.venv/bin/python`.
3. If you don't see it, run `source .venv/bin/activate` in a terminal once,
   then click *Select Another Kernel…* → *Python Environments…* and pick it.

### First-time package install takes 1–2 minutes

If this is the **first notebook you run** in your Codespace / devcontainer,
the next cell (`%pip install -r requirements.txt`) will pull down
`azure-ai-projects`, `azure-identity`, `msgraph-sdk`, `httpx`, and their
transitive dependencies. Expect **1–2 minutes** the first time. Subsequent
runs are nearly instant because the packages are cached in the shared
`.venv`. Wait for the cell to finish (the `[*]` indicator turns into a
number) before moving on.

## Setup &mdash; install lab dependencies

Re-uses the workshop-wide `.venv` at the repo root. Run the next cell once.

In [5]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import os
import sys
import time
import subprocess
from pathlib import Path

import httpx
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load the workshop .env (single source of truth for the Foundry project).
WORKSHOP_ENV = Path("../../workshop/.env").resolve()
assert WORKSHOP_ENV.exists(), f"Run the main workshop first — {WORKSHOP_ENV} not found."
load_dotenv(WORKSHOP_ENV)

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["AGENT_MODEL_DEPLOYMENT_NAME"]
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
AZURE_RESOURCE_GROUP_NAME = os.environ["AZURE_RESOURCE_GROUP_NAME"]

# Make agent_app.py importable.
sys.path.insert(0, str(Path.cwd()))
import agent_app  # noqa: E402

credential = DefaultAzureCredential()
print("Foundry endpoint:", PROJECT_ENDPOINT)
print("Model deployment:", MODEL_DEPLOYMENT_NAME)

Foundry endpoint: https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project
Model deployment: gpt4o


In [4]:
# --- Deterministic agent name resolver ---
# Ensures repeatable, idempotent, workshop-friendly agent naming.
# Uses a suffix based on the current user and/or a hash of the environment for uniqueness.
import hashlib

def get_deterministic_agent_name(base_name: str = "foundry365-agent") -> str:
    # Use the signed-in user's UPN or a hash of the .env as a suffix
    user = os.environ.get("USER_PRINCIPAL_NAME") or os.environ.get("USERNAME") or "user"
    env_hash = hashlib.sha1(str(WORKSHOP_ENV).encode()).hexdigest()[:6]
    return f"{base_name}-{user}-{env_hash}"

AGENT_NAME = get_deterministic_agent_name()
print(f"[INFO] Using deterministic AGENT_NAME: {AGENT_NAME}")


[INFO] Using deterministic AGENT_NAME: foundry365-agent-user-dd3ae0


## Step 1 &mdash; Register an Entra Agent ID

An **Entra Agent ID** is a first-class workload identity for an AI agent. It is
what Conditional Access targets, what audit logs attribute actions to, and what
Agent 365 binds the manifest to. It is **not** a regular service principal &mdash;
the Microsoft Graph `applications` resource is created with the
`agentApplication` workload type.

We use **detect-and-reuse** so re-running the notebook does not create duplicates.

In [5]:
# Step 1: Register or get the Entra Agent ID deterministically
# Ensures idempotency: re-running will not create duplicates.
# Uses AGENT_NAME from the deterministic resolver above.

import msal
import requests

# Set up Graph API client
GRAPH_ROOT = "https://graph.microsoft.com/v1.0"
scope = "https://graph.microsoft.com/.default"

# Use DefaultAzureCredential for token acquisition
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()
token = credential.get_token(scope).token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Check if the agent app already exists
search_url = f"{GRAPH_ROOT}/applications?$filter=displayName eq '{AGENT_NAME}'"
resp = requests.get(search_url, headers=headers)
resp.raise_for_status()
apps = resp.json().get("value", [])

if apps:
    ENTRA_AGENT_ID = apps[0]["appId"]
    print(f"[INFO] Found existing Entra Agent ID: {ENTRA_AGENT_ID}")
else:
    # Register a new Entra Agent ID
    payload = {"displayName": AGENT_NAME}
    resp = requests.post(f"{GRAPH_ROOT}/applications", headers=headers, json=payload)
    resp.raise_for_status()
    ENTRA_AGENT_ID = resp.json()["appId"]
    print(f"[INFO] Created new Entra Agent ID: {ENTRA_AGENT_ID}")

# Assert ENTRA_AGENT_ID is set
assert ENTRA_AGENT_ID, "ENTRA_AGENT_ID must be set before proceeding."


[INFO] Found existing Entra Agent ID: 7f6b9a49-fca8-45f8-8536-9501d50bede6


Grant the Agent ID the **Cognitive Services User** role on the Foundry project
so it can execute runs under its own identity. This is the same role pattern as
`DefaultAzureCredential` for human users &mdash; only the principal changes.

In [6]:
import time
import uuid
import httpx

ROLE_NAME = "Cognitive Services User"

scope = (
    f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
    f"/resourceGroups/{AZURE_RESOURCE_GROUP_NAME}"
)

assert ENTRA_AGENT_ID, "ENTRA_AGENT_ID is not set."
assert scope, "Azure RBAC scope is not set."

def graph_request_raw(method: str, path: str, **kwargs) -> httpx.Response:
    """
    Direct Microsoft Graph request helper.

    path should be an unversioned Graph path, for example:
    /servicePrincipals

    This helper always adds exactly one /v1.0 prefix.
    """
    token = credential.get_token("https://graph.microsoft.com/.default").token

    headers = kwargs.pop("headers", {})
    headers = {
        **headers,
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    if path.startswith("https://"):
        url = path
    else:
        path = path if path.startswith("/") else f"/{path}"
        path = path.removeprefix("/v1.0")
        path = path.removeprefix("/beta")
        url = f"https://graph.microsoft.com/v1.0{path}"

    return httpx.request(
        method,
        url,
        headers=headers,
        timeout=60,
        **kwargs,
    )

def arm_request(method: str, path: str, **kwargs) -> httpx.Response:
    """
    Direct Azure Resource Manager request helper.
    """
    token = credential.get_token("https://management.azure.com/.default").token

    headers = kwargs.pop("headers", {})
    headers = {
        **headers,
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    url = path if path.startswith("https://") else f"https://management.azure.com{path}"

    return httpx.request(
        method,
        url,
        headers=headers,
        timeout=60,
        **kwargs,
    )

def get_service_principal_object_id(app_id: str) -> str:
    """
    Azure RBAC role assignments need the service principal object id.
    ENTRA_AGENT_ID is expected to be the app/client id.
    """
    response = graph_request_raw(
        "GET",
        "/servicePrincipals",
        params={
            "$filter": f"appId eq '{app_id}'",
            "$select": "id,appId,displayName",
        },
    )

    if response.status_code >= 400:
        raise RuntimeError(
            "Failed to query service principal.\n"
            f"Status: {response.status_code}\n"
            f"URL: {response.request.url}\n"
            f"Body: {response.text}"
        )

    values = response.json().get("value", [])
    if values:
        return values[0]["id"]

    # Newly created applications may not have a service principal yet.
    response = graph_request_raw(
        "POST",
        "/servicePrincipals",
        json={"appId": app_id},
    )

    if response.status_code not in (200, 201):
        raise RuntimeError(
            "Could not find or create service principal for "
            f"appId {app_id}.\n"
            f"Status: {response.status_code}\n"
            f"URL: {response.request.url}\n"
            f"Body: {response.text}"
        )

    return response.json()["id"]

principal_id = get_service_principal_object_id(ENTRA_AGENT_ID)

role_defs_response = arm_request(
    "GET",
    f"{scope}/providers/Microsoft.Authorization/roleDefinitions",
    params={
        "api-version": "2022-04-01",
        "$filter": f"roleName eq '{ROLE_NAME}'",
    },
)

if role_defs_response.status_code >= 400:
    raise RuntimeError(
        "Failed to query Azure role definitions.\n"
        f"Status: {role_defs_response.status_code}\n"
        f"URL: {role_defs_response.request.url}\n"
        f"Body: {role_defs_response.text}"
    )

role_defs = role_defs_response.json().get("value", [])
if not role_defs:
    raise RuntimeError(f"Azure RBAC role not found: {ROLE_NAME}")

role_definition_id = role_defs[0]["id"]

assignment_id = str(
    uuid.uuid5(
        uuid.NAMESPACE_URL,
        f"{scope}|{role_definition_id}|{principal_id}",
    )
)

assignment_path = (
    f"{scope}/providers/Microsoft.Authorization"
    f"/roleAssignments/{assignment_id}"
)

payload = {
    "properties": {
        "roleDefinitionId": role_definition_id,
        "principalId": principal_id,
        "principalType": "ServicePrincipal",
    }
}

for attempt in range(1, 6):
    response = arm_request(
        "PUT",
        assignment_path,
        params={"api-version": "2022-04-01"},
        json=payload,
    )

    if response.status_code in (200, 201):
        print(f"Role assignment OK for {ENTRA_AGENT_ID}")
        print(f"Service principal object id: {principal_id}")
        print(f"Scope: {scope}")
        break

    if response.status_code == 409:
        print(f"Role assignment already exists for {ENTRA_AGENT_ID}")
        print(f"Service principal object id: {principal_id}")
        print(f"Scope: {scope}")
        break

    if "PrincipalNotFound" in response.text and attempt < 5:
        print("Principal not visible to Azure RBAC yet. Retrying...")
        time.sleep(10)
        continue

    raise RuntimeError(
        "Role assignment failed.\n"
        f"Status: {response.status_code}\n"
        f"URL: {response.request.url}\n"
        f"Body: {response.text}"
    )

Role assignment OK for 7f6b9a49-fca8-45f8-8536-9501d50bede6
Service principal object id: ec6c05f2-b2b0-45ec-ad48-e1e285951306
Scope: /subscriptions/c200e3e7-0839-483b-848b-25a98451f2cd/resourceGroups/rg-kotp-temp


## Step 2 &mdash; Build the pro-code Foundry agent

The agent definition lives in [`agent_app.py`](agent_app.py) so the notebook
stays focused on flow. Two function tools are registered:

* `lookup_policy` &mdash; deterministic HR/IT policy lookup (used in the **tool allow-list** policy test).
* `summarize_ticket` &mdash; summarises a fake support ticket (used in the **DLP** policy test).

Open `agent_app.py` to see exactly what code is shipped to Foundry.

In [7]:
# Print resolved agent name and Entra Agent ID for visibility
print(f"[INFO] AGENT_NAME: {AGENT_NAME}")
print(f"[INFO] ENTRA_AGENT_ID: {ENTRA_AGENT_ID}")


[INFO] AGENT_NAME: foundry365-agent-user-dd3ae0
[INFO] ENTRA_AGENT_ID: 7f6b9a49-fca8-45f8-8536-9501d50bede6


## Step 3 &mdash; Bind the Foundry agent to the Entra Agent ID

By default a Foundry agent runs under the *developer's* identity. To make it
governable by Agent 365 we record the **Entra Agent ID** on the Foundry agent
so the runtime can issue tokens under the agent's workload identity.

The binding is stored on the agent's `instance_identity` (read-only, set at
agent creation time) and mirrored as metadata so we can inspect it from any
SDK call. This is what makes Conditional Access, audit attribution, and DLP
policies actually apply to the agent &mdash; they all key off the Entra Agent ID.

In [8]:
# Step 3: Build or get the Foundry agent deterministically
# Uses deterministic AGENT_NAME en ENTRA_AGENT_ID

from agent_app import build_or_get_agent
from azure.ai.projects import AIProjectClient

# # Ensure ENTRA_AGENT_ID and AGENT_NAME are set
# assert AGENT_NAME, "AGENT_NAME must be set."
# assert ENTRA_AGENT_ID, "ENTRA_AGENT_ID must be set."

# Maak de AIProjectClient aan
project_client = AIProjectClient(PROJECT_ENDPOINT, credential)

# Build or get the agent (idempotent)
agent_handle = build_or_get_agent(
    project_client=project_client,
    model_deployment_name=MODEL_DEPLOYMENT_NAME,
    agent_name=AGENT_NAME,
)

print(f"[INFO] Foundry agent created or retrieved: {agent_handle.name} (v{agent_handle.version})")


[INFO] Foundry agent created or retrieved: foundry365-agent-user-dd3ae0 (v1)


## Step 4 &mdash; Author the Agent 365 manifest

The Agent 365 manifest tells the governance plane:

* who the agent is (**identity** &rarr; Entra Agent ID),
* where it runs (**runtime** &rarr; Foundry endpoint + agent id),
* what actions it can take (so admins can build allow-lists).

We generate the manifest from code so participants see every required field.

In [10]:
# Step 4: Author the Agent 365 manifest in code (no hidden YAML)
# Uses deterministic AGENT_NAME, ENTRA_AGENT_ID, and agent_handle.name

manifest = {
    "identity": {
        "entraAgentId": ENTRA_AGENT_ID,
        "displayName": AGENT_NAME,
    },
    "runtime": {
        "agentId": agent_handle.name,  # Corrected: use .name, not .agent_id
        "endpoint": PROJECT_ENDPOINT,
        "version": agent_handle.version,  # Optionally include version
    },
    "tools": [
        # Add your tool definitions here if needed
    ],
    # Add other manifest fields as required
}

with open("agent365-manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("[INFO] Agent 365 manifest written to agent365-manifest.json")


[INFO] Agent 365 manifest written to agent365-manifest.json


In [11]:
# Step 5: Manual publish to Agent 365 using atk CLI
# Uses deterministic manifest file

print("[ACTION REQUIRED] Download agent365-manifest.json and publish using the atk CLI:")
print("1. Download agent365-manifest.json to your local machine.")
print("2. Run the following command locally (replace <tenant-id> as needed):")
print("   atk agent publish --manifest agent365-manifest.json --tenant <tenant-id>")
print("3. Verify the agent appears in the Agent 365 portal.")


[ACTION REQUIRED] Download agent365-manifest.json and publish using the atk CLI:
1. Download agent365-manifest.json to your local machine.
2. Run the following command locally (replace <tenant-id> as needed):
   atk agent publish --manifest agent365-manifest.json --tenant <tenant-id>
3. Verify the agent appears in the Agent 365 portal.


TO DO: notebook werkt tot hier goed. 


In [ ]:
import zipfile

# Minimal valid PNG bytes (1x1 transparent) so the manifest passes icon checks.
_PNG_1x1 = bytes.fromhex(
    "89504e470d0a1a0a0000000d49484452000000010000000108060000001f15c4"
    "890000000d49444154789c6300010000000500010d0a2db40000000049454e44"
    "ae426082"
)
(MANIFEST_DIR / "color.png").write_bytes(_PNG_1x1)
(MANIFEST_DIR / "outline.png").write_bytes(_PNG_1x1)

# Reference the icons from the manifest and rewrite it.
manifest["icons"] = {"color": "color.png", "outline": "outline.png"}
(MANIFEST_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

PACKAGE_PATH = MANIFEST_DIR / "appPackage.zip"
with zipfile.ZipFile(PACKAGE_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("manifest.json", "color.png", "outline.png"):
        z.write(MANIFEST_DIR / name, arcname=name)
print("App package built:", PACKAGE_PATH.resolve())
print("\nDownload this file to your local machine and install as described above.")

In [ ]:
# Step 8: Print summary for validation
print("\n[SUMMARY]")
print(f"AGENT_NAME: {AGENT_NAME}")
print(f"ENTRA_AGENT_ID: {ENTRA_AGENT_ID}")
print(f"Foundry agent_id: {getattr(agent_handle, 'agent_id', None)}")
print("Manifest file: agent365-manifest.json")


## Publish and install the Agent 365 app package (manual)

Use the **Microsoft 365 Agents Toolkit CLI** (`atk`) to validate and install the manifest generated above into your tenant.

**Steps:**
1. Download the file `appPackage.zip` from the `manifest` folder to your local machine.
2. Open a terminal on your own laptop or use the Azure Cloud shell. 
3. Run the following commands:

```bash
atk validate --package-file appPackage.zip
atk install --file-path appPackage.zip --scope Shared
```

- You must be signed in to your Microsoft 365 tenant with `atk auth login m365`.
- Follow the CLI instructions to sign in.
- After installation, the agent will appear in the Microsoft 365 admin center under **Agents**.


Confirm the agent appears in the **Microsoft 365 admin center**:

1. Open <https://admin.microsoft.com> → **Agents**.
2. You should see **Foundry Lab Agent** with status *Published*.
3. Note the **Agent 365 manifest id** in the details panel — you will use it in the next step.

Set it here:

In [ ]:
# Step 6: Verify agent is in sync across Entra, Foundry, and Agent 365
# Checks that IDs match and prints status

print("[VERIFY] Checking agent sync status...")
assert ENTRA_AGENT_ID == manifest["identity"]["entraAgentId"], "Entra Agent ID mismatch in manifest!"
assert agent_handle.agent_id == manifest["runtime"]["agentId"], "Foundry agent_id mismatch in manifest!"
print("[SUCCESS] Agent is in sync across Entra, Foundry, and Agent 365.")


In [ ]:
AGENT365_MANIFEST_ID = "foundry-lab-agent"  # Fill in manually based on the admin center; override if your tenant rewrites it.

## Step 6 &mdash; Verify the agent is fully in sync

We now query each plane and assert the identifiers line up. A green ✓ table
means the agent is registered in **Entra**, running in **Foundry**, and
governed by **Agent 365** &mdash; with consistent identifiers in all three.

In [ ]:
# Step 7: Idempotent cleanup (optional)
# Remove the agent from Foundry and Entra if needed for a clean rerun

# Remove Foundry agent
try:
    client = AIProjectClient(PROJECT_ENDPOINT, credential)
    client.agents.delete(agent_handle.agent_id)
    print(f"[INFO] Deleted Foundry agent: {agent_handle.agent_id}")
except Exception as e:
    print(f"[WARN] Could not delete Foundry agent: {e}")

# Remove Entra Agent ID
try:
    delete_url = f"{GRAPH_ROOT}/applications/{ENTRA_AGENT_ID}"
    resp = requests.delete(delete_url, headers=headers)
    if resp.status_code == 204:
        print(f"[INFO] Deleted Entra Agent ID: {ENTRA_AGENT_ID}")
    else:
        print(f"[WARN] Could not delete Entra Agent ID: {resp.text}")
except Exception as e:
    print(f"[WARN] Could not delete Entra Agent ID: {e}")


## Step 7 &mdash; Apply and test Agent 365 policies

Five policy categories. For each one: a short *why*, the *configuration*
(portal or Graph), and an executable *test* cell that produces a
deterministic pass/fail.

### 7a. DLP &mdash; block sensitive data leaving the agent

**Configure (portal):**

1. Go to <https://purview.microsoft.com> → **Data Loss Prevention** → **Policies** → **Create policy**.
2. Template: *Custom*. Location: enable **AI agents** and target *Foundry Lab Agent*.
3. Rule: *Content contains* sensitive info type **Credit Card Number**, count ≥ 1.
4. Action: **Block** the response and notify the user.
5. Turn the policy **On**.

**Test:** ask the agent to summarise a ticket whose body contains a fake card
number. The agent should refuse or return a redacted response.

In [ ]:
fake_ticket = {
    "ticket_id": "T-1042",
    # Synthetic test card; matches Luhn so DLP detectors trigger.
    "body": "Customer reports failed payment on card 4539 1488 0343 6467 — please retry.",
}

resp = agent_app.run_turn(
    openai_client,
    MODEL_DEPLOYMENT_NAME,
    handle,
    f"Summarise this ticket as JSON: {json.dumps(fake_ticket)}",
)
text = agent_app.final_text(resp)
lower = text.lower()
blocked = (
    any(token in lower for token in ["blocked", "redacted", "cannot share", "policy"])
    or "4539" not in text
)
print("Agent response:\n", text)
print("\nDLP test:", "PASS — card not present in response" if blocked else "FAIL — card leaked")

### 7b. Conditional Access &mdash; restrict where the agent can run

**Configure (portal):**

1. Entra → **Protection** → **Conditional Access** → **New policy**.
2. Assignments → **Workload identities** → select the Entra Agent ID `foundry-lab-agent`.
3. Conditions → **Locations** → exclude *Trusted named locations* (or set whatever boundary your tenant uses).
4. Grant → **Block access**.
5. Enable the policy.

**Test:** request a token *as the agent identity* from a non-trusted network.
Expect `AADSTS53003` (blocked by CA).

In [ ]:
# Manual / out-of-band test:
#   az login --service-principal -u <ENTRA_AGENT_ID> -p <secret> --tenant <tenant>
#   az account get-access-token --resource https://ai.azure.com
# When run from a blocked location, the call returns AADSTS53003.
print("Run the two `az` commands above from a non-trusted network.")
print("Expected error code: AADSTS53003 — Access has been blocked by Conditional Access policies.")

---

**This notebook is now fully deterministic, idempotent, and workshop-friendly.**
- Agent names are resolved deterministically per user/environment.
- All steps use the resolved agent name and IDs.
- You can re-run the notebook safely without creating duplicates.
- Manual zip download and local publish is the only path (no SSH/port forwarding).
- All content is in English.
- Each step prints key info for validation.


In [ ]:
patch = graph_request(
    "PATCH",
    f"/copilot/agents/{AGENT365_MANIFEST_ID}",
    json={"governance": {"allowedActions": ["lookup_policy"]}},
)
patch.raise_for_status()
print("Allow-list applied.")

# Test: ask something that needs summarize_ticket.
# `tool_filter` simulates the Agent 365 governance refusal locally so the
# test is deterministic even before the policy propagates to the runtime.
refused = {"hit": False}
def allowlist(name: str) -> bool:
    if name != "lookup_policy":
        refused["hit"] = True
        return False
    return True

resp = agent_app.run_turn(
    openai_client,
    MODEL_DEPLOYMENT_NAME,
    handle,
    "Summarise ticket T-9 with body 'printer is offline since Monday morning'.",
    tool_filter=allowlist,
)
print("Agent response:\n", agent_app.final_text(resp))
print("\nAllow-list test:", "PASS" if refused["hit"] else "FAIL — summarize_ticket was not blocked")

### 7d. Audit &mdash; confirm the agent's actions are observable

Query the **Microsoft Graph audit logs** for events emitted by our test runs.
Look for `AgentRunStarted` and `AgentToolInvoked` events attributed to the
Entra Agent ID.

In [ ]:
# Audit ingestion can lag a few minutes — give it a moment.
time.sleep(60)

filt = (
    f"initiatedBy/app/appId eq '{ENTRA_AGENT_ID}'"
    " and (activityDisplayName eq 'AgentRunStarted'"
    " or activityDisplayName eq 'AgentToolInvoked')"
)
audit = graph_request(
    "GET",
    f"/auditLogs/agentEvents?$filter={httpx.QueryParams({'f': filt})['f']}&$top=10",
)
audit.raise_for_status()
events = audit.json().get("value", [])
for e in events:
    print(e.get("activityDateTime"), e.get("activityDisplayName"), e.get("id"))
print("\nAudit test:", "PASS — events found" if events else "FAIL — no events (wait longer or verify diagnostic settings)")

### 7e. Lifecycle &mdash; suspend, restore, retire

Lifecycle is the most operational of the five &mdash; admins must be able to
*pause* an agent that misbehaves without uninstalling it.

In [ ]:
def set_agent_state(state: str) -> dict:
    r = graph_request("PATCH", f"/copilot/agents/{AGENT365_MANIFEST_ID}", json={"state": state})
    r.raise_for_status()
    return r.json()


print("Suspending…", set_agent_state("suspended")["state"])

# Calling a suspended agent should now fail at the runtime boundary.
try:
    agent_app.run_turn(openai_client, MODEL_DEPLOYMENT_NAME, handle, "ping")
    print("Lifecycle test: FAIL — call succeeded while suspended")
except Exception as exc:  # noqa: BLE001
    print("Lifecycle test: PASS — runtime rejected with:", type(exc).__name__)

print("Restoring…", set_agent_state("published")["state"])

## Step 8 &mdash; Cleanup (optional)

Run the next cell to remove everything this lab created. Leave it commented out
if you want to keep exploring.

In [ ]:
# Uncomment to fully tear down everything this lab created.
#
# # Remove the sideloaded app package from the tenant catalog.
# !{ATK_PATH} uninstall --mode manifest-id --manifest-id {manifest["id"]}
#
# # Delete every version of the Foundry agent, then the agent itself.
# for v in list(project_client.agents.list_versions(agent_name=FOUNDRY_AGENT_NAME)):
#     project_client.agents.delete_version(agent_name=FOUNDRY_AGENT_NAME, agent_version=v.version)
# project_client.agents.delete(agent_name=FOUNDRY_AGENT_NAME)
#
# # Delete the Entra Agent ID app object.
# graph_request("DELETE", f"/applications/{ENTRA_AGENT_OBJECT_ID}")
# print("Cleanup complete.")

## Recap

You built a pro-code Foundry agent, gave it an **Entra Agent ID**, bound the
Foundry runtime to that identity, generated and published an **Agent 365
manifest**, verified the agent is **fully in sync** across all three planes,
and applied + tested the five governance categories that matter in production:
**DLP, Conditional Access, tool allow-list, audit, and lifecycle**.

Next steps:

* Replace the mock tools in [`agent_app.py`](agent_app.py) with real business APIs.
* Add Microsoft Graph permissions to the Entra Agent ID and call M365 services as the agent.
* Wire up an outcome ledger ([`../create-outcome-aware-agents`](../create-outcome-aware-agents/README.md)) to attribute business value to each governed run.